# Setup local Django (SQLite) no Windows
Este notebook executa o fluxo solicitado: versões, `INSTALAR.bat` com fallback, `check`, subida do servidor e teste HTTP local, com relatório objetivo e erros completos.

In [ ]:
import os
import sys
import time
import json
import traceback
import subprocess
from pathlib import Path
from urllib import request, error

workspace = Path(r"c:\logistica_project")
os.chdir(workspace)

comandos_executados = []
resultado_etapas = {}
erros_relevantes = []


def _format_result(res):
    return {
        "returncode": res.get("returncode"),
        "stdout": res.get("stdout", ""),
        "stderr": res.get("stderr", ""),
        "exception": res.get("exception", ""),
        "traceback": res.get("traceback", ""),
    }


def run_cmd(cmd, step_name, timeout=1800, shell=False):
    cmd_display = cmd if isinstance(cmd, str) else " ".join(cmd)
    comandos_executados.append(cmd_display)
    try:
        completed = subprocess.run(
            cmd,
            cwd=str(workspace),
            capture_output=True,
            text=True,
            timeout=timeout,
            shell=shell,
        )
        ok = completed.returncode == 0
        res = {
            "returncode": completed.returncode,
            "stdout": completed.stdout,
            "stderr": completed.stderr,
            "exception": "",
            "traceback": "",
        }
    except Exception as exc:
        ok = False
        res = {
            "returncode": None,
            "stdout": "",
            "stderr": "",
            "exception": repr(exc),
            "traceback": traceback.format_exc(),
        }

    resultado_etapas[step_name] = "OK" if ok else "FALHA"
    if not ok:
        erros_relevantes.append({
            "etapa": step_name,
            "comando": cmd_display,
            **_format_result(res),
        })
    return ok, res


# 1) Verificar versões
ok_py, res_py = run_cmd(["python", "--version"], "1.1 python --version", timeout=120)
ok_pip, res_pip = run_cmd(["python", "-m", "pip", "--version"], "1.2 python -m pip --version", timeout=120)

# 2) Tentar INSTALAR.bat; fallback se falhar
instalar_ok, res_instalar = run_cmd(["cmd", "/c", "INSTALAR.bat"], "2. INSTALAR.bat", timeout=3600)

venv_python = workspace / ".venv" / "Scripts" / "python.exe"
venv_pip = workspace / ".venv" / "Scripts" / "pip.exe"

if not instalar_ok:
    run_cmd(["python", "-m", "venv", ".venv"], "2.fallback.1 python -m venv .venv", timeout=600)
    run_cmd([str(venv_python), "-m", "pip", "install", "--upgrade", "pip"], "2.fallback.2 .venv\\Scripts\\python -m pip install --upgrade pip", timeout=1200)
    run_cmd([str(venv_pip), "install", "-r", "requirements.txt"], "2.fallback.3 .venv\\Scripts\\pip install -r requirements.txt", timeout=3600)
    run_cmd([str(venv_python), "manage.py", "migrate"], "2.fallback.4 .venv\\Scripts\\python manage.py migrate", timeout=1800)
else:
    # Mesmo com INSTALAR.bat OK, garante etapa de migrate para setup SQLite
    migrate_cmd = [str(venv_python), "manage.py", "migrate"] if venv_python.exists() else ["python", "manage.py", "migrate"]
    run_cmd(migrate_cmd, "2.extra manage.py migrate", timeout=1800)

# 3) manage.py check via .venv
check_cmd = [str(venv_python), "manage.py", "check"]
check_ok, res_check = run_cmd(check_cmd, "3. .venv\\Scripts\\python manage.py check", timeout=600)

# 4) Iniciar servidor em background porta 8000
server_status = "não rodando"
server_evidencia = ""
server_log_path = workspace / ".tmp_runserver_8000.log"
server_proc = None

try:
    with open(server_log_path, "w", encoding="utf-8") as logf:
        server_proc = subprocess.Popen(
            [str(venv_python), "manage.py", "runserver", "8000"],
            cwd=str(workspace),
            stdout=logf,
            stderr=logf,
            text=True,
        )
    comandos_executados.append(f"{venv_python} manage.py runserver 8000 (background)")
    time.sleep(6)
    poll = server_proc.poll()
    if poll is None:
        server_status = "rodando"
        server_evidencia = f"Processo ativo sem crash imediato. PID={server_proc.pid}. Log: {server_log_path.name}"
        resultado_etapas["4. runserver 8000 background"] = "OK"
    else:
        resultado_etapas["4. runserver 8000 background"] = "FALHA"
        log_content = server_log_path.read_text(encoding="utf-8", errors="replace") if server_log_path.exists() else "(log não encontrado)"
        server_evidencia = f"Processo encerrou cedo com code={poll}. Log completo:\n{log_content}"
        erros_relevantes.append({
            "etapa": "4. runserver 8000 background",
            "comando": f"{venv_python} manage.py runserver 8000",
            "returncode": poll,
            "stdout": "",
            "stderr": log_content,
            "exception": "",
            "traceback": "",
        })
except Exception as exc:
    resultado_etapas["4. runserver 8000 background"] = "FALHA"
    tb = traceback.format_exc()
    server_evidencia = f"Falha ao iniciar servidor: {exc}\n{tb}"
    erros_relevantes.append({
        "etapa": "4. runserver 8000 background",
        "comando": f"{venv_python} manage.py runserver 8000",
        "returncode": None,
        "stdout": "",
        "stderr": "",
        "exception": repr(exc),
        "traceback": tb,
    })

# 5) Checar HTTP local
http_status = None
http_detail = ""
try:
    req = request.Request("http://127.0.0.1:8000", method="GET")
    with request.urlopen(req, timeout=10) as resp:
        http_status = resp.status
        body_preview = resp.read(400).decode("utf-8", errors="replace")
        http_detail = f"HTTP {http_status}. Prévia da resposta: {body_preview[:250]}"
    resultado_etapas["5. HTTP localhost:8000"] = "OK"
except Exception as exc:
    tb = traceback.format_exc()
    http_detail = f"Falha HTTP: {exc}\n{tb}"
    resultado_etapas["5. HTTP localhost:8000"] = "FALHA"
    erros_relevantes.append({
        "etapa": "5. HTTP localhost:8000",
        "comando": "GET http://127.0.0.1:8000",
        "returncode": None,
        "stdout": "",
        "stderr": "",
        "exception": repr(exc),
        "traceback": tb,
    })

# Consolidar saída objetiva
print("Comandos executados")
for cmd in comandos_executados:
    print(f"- {cmd}")

print("\nResultado por etapa")
for etapa, status in resultado_etapas.items():
    print(f"- {etapa}: {status}")

print("\nServidor")
print(f"- {server_status} | {server_evidencia}")
print(f"- HTTP localhost:8000 | {http_detail}")

print("\nPróximo comando que o usuário deve rodar")
if server_status != "rodando" or resultado_etapas.get("5. HTTP localhost:8000") != "OK":
    print(r"- .venv\Scripts\python manage.py runserver 8000")
else:
    print("- (não necessário)")

print("\nErros relevantes completos")
if not erros_relevantes:
    print("- Nenhum erro relevante.")
else:
    for i, err in enumerate(erros_relevantes, 1):
        print(f"\n[Erro {i}] Etapa: {err['etapa']}")
        print(f"Comando: {err['comando']}")
        print(f"Return code: {err['returncode']}")
        if err.get("stdout"):
            print("--- STDOUT ---")
            print(err["stdout"])
        if err.get("stderr"):
            print("--- STDERR ---")
            print(err["stderr"])
        if err.get("exception"):
            print("--- EXCEPTION ---")
            print(err["exception"])
        if err.get("traceback"):
            print("--- TRACEBACK ---")
            print(err["traceback"])
